In [2]:
!pip install -q anthropic sentence-transformers groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.2 MB/s eta 0:00:00


In [3]:
import os, getpass

try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = api_key

from groq import Groq
client = Groq()
JUDGE_MODEL = "openai/gpt-oss-120b" # Using a Groq-compatible model
print("Client ready.")

Client ready.


In [4]:
good_example = {
    "question": "What is the deductible for collision coverage?",
    "contexts": [
        "Collision Coverage: The Company will pay for direct and accidental physical loss "
        "to your covered auto caused by collision, subject to a $500 deductible per "
        "occurrence. Coverage applies regardless of fault.",
    ],
    "answer": "The deductible for collision coverage is $500 per occurrence.",
    "ground_truth": "The deductible for collision coverage is $500 per occurrence.",
}

hallucinated_example = {
    "question": "How much does the company pay for a rental car after a collision?",
    "contexts": [
        "Rental Reimbursement: If your covered auto is out of service due to a covered "
        "collision or comprehensive loss, the Company will reimburse rental costs up to "
        "$40 per day for a maximum of 30 days.",
    ],
    "answer": (
        "The company reimburses rental costs up to $40 per day for up to 30 days, and also "
        "provides a one-time $200 loyalty bonus for customers with more than 5 years of "
        "continuous coverage."
    ),  # the $200 loyalty bonus is fabricated — not in the context at all
    "ground_truth": "The company reimburses rental costs up to $40 per day for a maximum of 30 days.",
}

bad_retrieval_example = {
    "question": "What is the dollar threshold for escalating a claim to a Senior Claims Adjuster?",
    "contexts": [
        # Wrong chunks retrieved — the real escalation-threshold clause was never fetched
        "Water Damage Exclusion: Damage caused by flood, surface water, or sewer backup is "
        "excluded from standard coverage. Separate flood insurance must be purchased.",
        "Roadside Assistance: The Company will reimburse reasonable towing and labor costs "
        "up to $100 per disablement, limited to four disablements per policy period.",
    ],
    "answer": "Claims over $5,000 must be escalated to a Senior Claims Adjuster.",  # fabricated number
    "ground_truth": "Claims over $10,000 must be escalated to a Senior Claims Adjuster before settlement is authorized.",
}

examples = {
    "good_example": good_example,
    "hallucinated_example": hallucinated_example,
    "bad_retrieval_example": bad_retrieval_example,
}
print(f"Loaded {len(examples)} evaluation examples.")

Loaded 3 evaluation examples.


In [5]:
def decompose_into_claims(answer: str) -> list[str]:
    prompt = (
        "Break the following answer into a numbered list of atomic factual claims — each "
        "claim should be a single, independently checkable statement. Respond with ONLY the "
        "numbered list, one claim per line, no other text.\n\n"
        f"Answer: {answer}"
    )
    resp = client.chat.completions.create(model=JUDGE_MODEL, max_tokens=300,
                                   messages=[{"role": "user", "content": prompt}])
    lines = resp.choices[0].message.content.strip().split("\n")
    claims = [l.split(".", 1)[-1].strip() for l in lines if l.strip()]
    return claims


def claim_supported_by_context(claim: str, contexts: list[str]) -> bool:
    context_text = "\n\n".join(contexts)
    prompt = (
        "Given the CONTEXT and a CLAIM, respond with only one word: YES if the claim is "
        "directly supported by the context, or NO if it is not supported or not mentioned.\n\n"
        f"CONTEXT:\n{context_text}\n\nCLAIM: {claim}"
    )
    resp = client.chat.completions.create(model=JUDGE_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip().upper().startswith("YES")


def faithfulness(example: dict) -> dict:
    claims = decompose_into_claims(example["answer"])
    verdicts = [claim_supported_by_context(c, example["contexts"]) for c in claims]
    score = sum(verdicts) / len(verdicts) if verdicts else 0.0
    return {"score": score, "claims": list(zip(claims, verdicts))}


# Run it on all three examples
for name, ex in examples.items():
    result = faithfulness(ex)
    print(f"=== {name} — Faithfulness: {result['score']:.2f} ===")
    for claim, supported in result["claims"]:
        print(f"  [{'OK' if supported else 'UNSUPPORTED'}] {claim}")
    print()

=== good_example — Faithfulness: 0.00 ===
  [UNSUPPORTED] The deductible for collision coverage is $500 per occurrence.

=== hallucinated_example — Faithfulness: 0.00 ===
  [UNSUPPORTED] The company reimburses rental costs up to $40 per day for up to 30 days.
  [UNSUPPORTED] The company provides a one-time $200 loyalty bonus for customers with more than 5 years of continuous coverage.

=== bad_retrieval_example — Faithfulness: 0.00 ===



In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def generate_synthetic_questions(answer: str, n: int = 3) -> list[str]:
    prompt = (
        f"Generate {n} different questions that the following ANSWER would be a good, "
        "direct response to. Respond with ONLY the questions, one per line, no numbering.\n\n"
        f"ANSWER: {answer}"
    )
    resp = client.chat.completions.create(model=JUDGE_MODEL, max_tokens=200,
                                   messages=[{"role": "user", "content": prompt}])
    return [q.strip() for q in resp.choices[0].message.content.strip().split("\n") if q.strip()]


def answer_relevancy(example: dict, n: int = 3) -> dict:
    synthetic_questions = generate_synthetic_questions(example["answer"], n=n)
    all_texts = [example["question"]] + synthetic_questions
    embeddings = embedder.encode(all_texts, normalize_embeddings=True)
    original_emb, synthetic_embs = embeddings[0], embeddings[1:]
    similarities = synthetic_embs @ original_emb
    return {"score": float(np.mean(similarities)), "synthetic_questions": synthetic_questions,
            "similarities": similarities.tolist()}


for name, ex in examples.items():
    result = answer_relevancy(ex)
    print(f"=== {name} — Answer Relevancy: {result['score']:.2f} ===")
    print(f"  Original question: {ex['question']}")
    for q, sim in zip(result["synthetic_questions"], result["similarities"]):
        print(f"  [{sim:.2f}] Reverse-engineered: {q}")
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== good_example — Answer Relevancy: 0.81 ===
  Original question: What is the deductible for collision coverage?
  [0.86] Reverse-engineered: What is the deductible amount for collision coverage on my auto insurance policy?
  [0.71] Reverse-engineered: How much do I need to pay out‑of‑pocket before my collision coverage applies?
  [0.86] Reverse-engineered: What is the per‑occurrence deductible for collision insurance?

=== hallucinated_example — Answer Relevancy: 0.36 ===
  Original question: How much does the company pay for a rental car after a collision?
  [0.46] Reverse-engineered: What is the daily rental reimbursement limit and the maximum number of days the company will cover?
  [0.27] Reverse-engineered: Does the company offer any bonus or incentive for customers who have been continuously covered for more than five years?
  [0.36] Reverse-engineered: Can you explain the rental cost reimbursement policy and any additional loyalty bonuses for long‑term policyholders?

=== bad_

In [7]:
def chunk_relevant_to_question(chunk: str, question: str) -> bool:
    prompt = (
        "Given a QUESTION and a retrieved CONTEXT CHUNK, respond with only one word: YES if "
        "the chunk contains information that helps answer the question, or NO if it is "
        "irrelevant.\n\n"
        f"QUESTION: {question}\n\nCONTEXT CHUNK: {chunk}"
    )
    resp = client.chat.completions.create(model=JUDGE_MODEL, max_tokens=5,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip().upper().startswith("YES")


def context_precision(example: dict) -> dict:
    relevance = [chunk_relevant_to_question(c, example["question"]) for c in example["contexts"]]
    if not any(relevance):
        return {"score": 0.0, "relevance": relevance}

    # Average Precision: at each relevant position k, compute precision@k, then average
    precisions_at_k = []
    num_relevant_so_far = 0
    for k, is_relevant in enumerate(relevance, start=1):
        if is_relevant:
            num_relevant_so_far += 1
            precisions_at_k.append(num_relevant_so_far / k)
    score = sum(precisions_at_k) / sum(relevance)
    return {"score": score, "relevance": relevance}

# C1, C2, C3, C4, C5
# Relevant ones = C1, C3, C5
# precisions_at_k = [1, 2/3, 3/5]


for name, ex in examples.items():
    result = context_precision(ex)
    print(f"=== {name} — Context Precision: {result['score']:.2f} ===")
    for chunk, is_rel in zip(ex["contexts"], result["relevance"]):
        print(f"  [{'RELEVANT' if is_rel else 'NOT RELEVANT'}] {chunk[:70]}...")
    print()

=== good_example — Context Precision: 0.00 ===
  [NOT RELEVANT] Collision Coverage: The Company will pay for direct and accidental phy...

=== hallucinated_example — Context Precision: 0.00 ===
  [NOT RELEVANT] Rental Reimbursement: If your covered auto is out of service due to a ...

=== bad_retrieval_example — Context Precision: 0.00 ===
  [NOT RELEVANT] Water Damage Exclusion: Damage caused by flood, surface water, or sewe...
  [NOT RELEVANT] Roadside Assistance: The Company will reimburse reasonable towing and ...

